# fuzzy-enigma: pricing path-dependent exotics from Python

This notebook walks through the Python bindings for the `fuzzy-enigma` Rust crate. We will:

1. Set up a Geometric Brownian Motion model and a Monte Carlo engine.
2. Cross-check the engine against the closed-form Black-Scholes price.
3. Price the full menu of exotics in a single table.
4. Plot Monte Carlo convergence for an Asian call.
5. Sweep the barrier level of an up-and-out call and compare to the vanilla limit.

## Building the extension module

The Python module is built from Rust via [maturin](https://www.maturin.rs/). From the repository root:

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install maturin numpy matplotlib jupyter ipykernel
maturin develop --release
```

`maturin develop` reads `pyproject.toml`, compiles the Rust crate with the `python` feature on, and installs the resulting extension module into the active venv as `fuzzy_enigma`.

## Running this notebook with the right kernel

The extension is built with PyO3's `abi3-py39` feature, so `maturin develop` produces a single `fuzzy_enigma.abi3.so` that loads on **any CPython 3.9 or newer** - it is not pinned to the interpreter that compiled it. A venv built on 3.14 is importable from a 3.10 or 3.12 kernel just as well.

What still matters is that the venv is on the kernel's path. The repository ships a kernel for exactly that:

```bash
.venv/bin/python -m ipykernel install --user \
    --name fuzzy-enigma --display-name 'Python 3 (fuzzy-enigma)'
```

This notebook is saved with that kernel selected, so it should just run. If you are on some other kernel, the first cell finds the project venv and appends it to `sys.path` for you. It *appends* rather than prepends on purpose: `numpy` and `matplotlib` are version-specific compiled extensions with no stable ABI, so putting a 3.14 venv ahead of a 3.12 kernel's own packages would shadow a working numpy with one that cannot load.


In [ ]:
import sys
import pathlib


def _ensure_fuzzy_enigma():
    """Make `fuzzy_enigma` importable from whichever kernel is running.

    The extension is built with PyO3's `abi3-py39` feature, so a single
    `fuzzy_enigma.abi3.so` loads on any CPython >= 3.9 rather than being
    pinned to the version that compiled it. That means simply putting the
    project venv on the path is enough, even from a different interpreter.

    The path entry is *appended*, never prepended: `numpy` and
    `matplotlib` are version-specific compiled extensions with no stable
    ABI, so a 3.14 venv at the front of a 3.12 path would shadow this
    kernel's working numpy with one it cannot load. Appending lets the
    kernel's own packages win and leaves the venv purely as a fallback for
    the module it alone provides.
    """
    try:
        import fuzzy_enigma  # noqa: F401
        return
    except ModuleNotFoundError:
        pass

    for root in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
        for site_dir in sorted((root / ".venv" / "lib").glob("python3.*/site-packages")):
            sys.path.append(str(site_dir))
            try:
                import fuzzy_enigma  # noqa: F401
                print(f"note: loaded fuzzy_enigma from {site_dir}")
                return
            except ImportError:
                sys.path.pop()

    raise ModuleNotFoundError(
        "fuzzy_enigma is not importable from this kernel, and no project "
        ".venv containing it was found.\n"
        f"  this kernel: Python {sys.version_info.major}.{sys.version_info.minor} "
        f"at {sys.executable}\n\n"
        "Build it from the repository root:\n"
        "    python3 -m venv .venv && source .venv/bin/activate\n"
        "    pip install maturin numpy matplotlib jupyter ipykernel\n"
        "    maturin develop --release\n\n"
        "The build needs Python >= 3.9; the resulting extension then works "
        "on any 3.9+ kernel."
    )


print("interpreter:", sys.executable, f"(Python {sys.version.split()[0]})")
_ensure_fuzzy_enigma()

import fuzzy_enigma as fe
import numpy as np
import matplotlib.pyplot as plt
from math import sqrt, log, exp, erf

print(f"fuzzy_enigma version: {fe.__version__}")


## 1. Model and engine

`GbmModel` is the only model bundled with the library: a single-factor GBM under the risk-neutral measure with constant rate, dividend yield, and volatility. `McEngine` is the Monte Carlo runner; the same engine is reused across every payoff.

In [ ]:
S0, r, q, sigma, T, steps = 100.0, 0.03, 0.0, 0.25, 1.0, 252

model = fe.GbmModel(spot=S0, r=r, q=q, sigma=sigma, t=T, steps=steps)
engine = fe.McEngine(paths=200_000, seed=42, antithetic=True, parallel=True)

print(model)
print(engine)

## 2. Sanity check against Black-Scholes

The vanilla European call has a closed form. The Monte Carlo price should land inside its 95% confidence interval. (If it does not, either the engine is broken or you got very unlucky.)

In [ ]:
def norm_cdf(x):
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))

def bs_call(S, K, r, q, sigma, T):
    d1 = (log(S / K) + (r - q + 0.5 * sigma ** 2) * T) / (sigma * sqrt(T))
    d2 = d1 - sigma * sqrt(T)
    return S * exp(-q * T) * norm_cdf(d1) - K * exp(-r * T) * norm_cdf(d2)

K = 100.0
analytic = bs_call(S0, K, r, q, sigma, T)
mc = engine.price_european(model, "call", K)
lo, hi = mc.confidence_95()

print(f"Black-Scholes:  {analytic:.4f}")
print(f"Monte Carlo:    {mc.price:.4f}  (95% CI [{lo:.4f}, {hi:.4f}])")
print(f"BS price inside CI? {lo <= analytic <= hi}")

## 3. Price the full menu of exotics

Every payoff goes through the same engine, so swapping models (e.g. plugging in a future Heston model) would not change a line below.

In [ ]:
products = [
    ("European call K=100",          engine.price_european(model, "call", 100.0)),
    ("Asian call K=100",             engine.price_asian(model, "call", 100.0)),
    ("Up-and-out call K=100 B=130",  engine.price_barrier(model, "call", "up_and_out", 100.0, 130.0)),
    ("Down-and-in put  K=100 B=80",  engine.price_barrier(model, "put",  "down_and_in", 100.0, 80.0)),
    ("Floating-strike lookback",     engine.price_lookback(model, "call")),
    ("Cliquet -3% / +5%",            engine.price_cliquet(model, 100.0, -0.03, 0.05)),
    ("Autocallable (quarterly)",     engine.price_autocallable(model, 100.0, 2.0, 100.0, 70.0, [63, 126, 189, 252])),
]

header = f"{'payoff':<32}{'price':>10}{'std-err':>10}{'95% CI':>22}"
print(header)
print('-' * len(header))
for name, res in products:
    lo, hi = res.confidence_95()
    print(f"{name:<32}{res.price:>10.4f}{res.std_error:>10.4f}    [{lo:>7.3f}, {hi:>7.3f}]")

## 4. Monte Carlo convergence

Standard error should shrink like $1/\sqrt{n}$. We re-price the Asian call across a logarithmic grid of path counts and plot the price with a 95% confidence band.

In [ ]:
n_paths_grid = [1_000, 2_500, 5_000, 10_000, 25_000, 50_000, 100_000, 200_000, 400_000]
prices, errs = [], []
for n in n_paths_grid:
    eng = fe.McEngine(paths=n, seed=42, antithetic=True, parallel=True)
    out = eng.price_asian(model, "call", 100.0)
    prices.append(out.price)
    errs.append(out.std_error)
prices, errs = np.array(prices), np.array(errs)

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(n_paths_grid, prices, yerr=1.96 * errs, fmt="o-", capsize=3, label="MC price ± 95% CI")
ax.set_xscale("log")
ax.set_xlabel("Monte Carlo paths")
ax.set_ylabel("Asian call price")
ax.set_title("Convergence of Asian call price")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 5. Barrier sweep: up-and-out call

As the barrier $B \to \infty$ the up-and-out call must converge to the vanilla European call (the barrier becomes impossible to hit). For low barriers most paths get knocked out and the option is nearly worthless.

In [ ]:
barriers = np.linspace(105, 250, 25)
prices, errs = [], []
for B in barriers:
    out = engine.price_barrier(model, "call", "up_and_out", 100.0, float(B))
    prices.append(out.price)
    errs.append(out.std_error)
prices, errs = np.array(prices), np.array(errs)

vanilla = engine.price_european(model, "call", 100.0).price

fig, ax = plt.subplots(figsize=(7, 4))
ax.fill_between(barriers, prices - 1.96 * errs, prices + 1.96 * errs, alpha=0.25, label="95% CI")
ax.plot(barriers, prices, "o-", label="up-and-out call")
ax.axhline(vanilla, color="C3", ls="--", label=f"vanilla European = {vanilla:.3f}")
ax.set_xlabel("barrier level B")
ax.set_ylabel("price")
ax.set_title("Up-and-out call price as function of barrier")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## Where to go from here

* **More models.** Implement the Rust `PathGenerator` trait (Heston, jump-diffusion, multi-asset basket) and re-expose it through `src/python.rs`. None of the payoffs need to change.
* **Greeks beyond the vanilla.** `engine.greeks_european(model, "call", K)` ships in the bindings, using bump-and-revalue under common random numbers. Exposing the same for the exotics is a small addition to `src/python.rs`; pathwise and likelihood-ratio estimators would be the next step up in accuracy.
* **Calibration.** Wrap any of the `price_*` methods in a `scipy.optimize.least_squares` call to fit volatility (or a Heston parameter set) to a market quote sheet.